In [ ]:
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

DATA_DIR = Path('examples/datasets/maneuvers_small')
OUT_DIR = DATA_DIR / 'gallery'
OUT_DIR.mkdir(parents=True, exist_ok=True)

manifest = json.loads((DATA_DIR / 'manifest.json').read_text())

def load_csv(path):
    import csv
    t, ax, ay, az, gx, gy, gz = [], [], [], [], [], [], []
    with open(path, 'r') as fh:
        reader = csv.DictReader(fh)
        for r in reader:
            t.append(float(r['t']))
            ax.append(float(r['ax']))
            ay.append(float(r['ay']))
            az.append(float(r['az']))
            gx.append(float(r['gx']))
            gy.append(float(r['gy']))
            gz.append(float(r['gz']))
    t = np.array(t)
    accel = np.vstack([ax, ay, az]).T
    gyro = np.vstack([gx, gy, gz]).T
    return t, accel, gyro

In [ ]:
# Plot each maneuver into a single page (use a grid to keep the notebook size reasonable)
for i, entry in enumerate(manifest):
    path = Path(entry['file'])
    segments = entry['segments']
    t, accel, gyro = load_csv(path)
    accel_mag = np.linalg.norm(accel, axis=1)
    gyro_mag = np.linalg.norm(gyro, axis=1)
    name = segments[0][2] if segments else path.stem

    fig, axs = plt.subplots(3, 1, figsize=(8, 6), sharex=True)
    axs[0].plot(t, accel_mag, label='|a|')
    axs[0].plot(t, accel[:, 0], label='ax', alpha=0.6)
    axs[0].plot(t, accel[:, 1], label='ay', alpha=0.6)
    axs[0].plot(t, accel[:, 2], label='az', alpha=0.6)
    axs[0].legend(loc='upper right', fontsize='small')
    axs[0].set_ylabel('accel (m/s^2)')

    axs[1].plot(t, gyro_mag, label='|ω|', color='C1')
    axs[1].plot(t, gyro[:, 0], label='gx', alpha=0.6, color='C2')
    axs[1].plot(t, gyro[:, 1], label='gy', alpha=0.6, color='C3')
    axs[1].plot(t, gyro[:, 2], label='gz', alpha=0.6, color='C4')
    axs[1].legend(loc='upper right', fontsize='small')
    axs[1].set_ylabel('gyro (rad/s)')

    # mark GT segment start/end if present
    if segments:
        s, e, lbl = segments[0]
        axs[0].axvspan(t[s], t[e-1] if e-1 < len(t) else t[-1], color='C0', alpha=0.08)
        axs[1].axvspan(t[s], t[e-1] if e-1 < len(t) else t[-1], color='C0', alpha=0.08)

    axs[2].plot(t, accel[:, 0], label='ax')
    axs[2].plot(t, accel[:, 1], label='ay')
    axs[2].plot(t, accel[:, 2], label='az')
    axs[2].set_ylabel('accel axes')
    axs[2].set_xlabel('time (s)')

    fig.suptitle(f'{i:02d} - {name}')
    out = OUT_DIR / f'{i:02d}_{name}.png'
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    fig.savefig(out, dpi=150)
    plt.close(fig)

print('Saved gallery images to', OUT_DIR)

Run `jupyter nbconvert --to html examples/notebooks/maneuver_gallery.ipynb`
to export an HTML file that CI can upload as an artifact.